In [ ]:
#| default_exp machine_learning.definition_and_notation_naming

# machine_learning.definition_and_notation_naming

> Functions for gathering and processing data to train and for using ML models to "name" definitions and notations

`trouver.machine_learning.tokenize.def_and_notat_token_classification` has functions for gathering and processing data to train and for using ML models to identify definitions and notations introduced in notes via token classification. Identified definition and notations are marked using HTML tags. It would be convenient to predict the "names" for these definitions and notations.

TODO: insert examples of definitions and notations with HTML tags and examples of what the "names" of these definitions and notation should be

In [ ]:
#| export
import copy
import random
from typing import Literal, Optional, TypedDict

import bs4
from bs4 import BeautifulSoup, Tag
from pathvalidate import sanitize_filename
from transformers import pipelines
import warnings

from trouver.helper.html import (
    remove_html_tags_in_text, add_HTML_tag_data_to_raw_text,
    StrAndHTMLTagsWithIndices, HTMLTagWithIndices)
from trouver.helper.latex.formatting import (
    fix_autogen_formatting
)
from trouver.helper.latex.processing import (
    correct_latex_syntax_error, _list_of_candidates_from_math_mode_strings, math_mode_string_is_syntactically_valid,
)
from trouver.helper.latex.augment import (
    random_char_modification,
    dollar_sign_manipulation, remove_math_keywords, random_word_removal, random_latex_command_removal,
    push_dollar_signs, augment_text,
    change_font_styles_at_random, change_greek_letters_at_random, remove_font_styles_at_random
)

from trouver.helper.latex.macros_and_commands import (math_mode_string_has_soft_or_hard_syntax_errors)
from trouver.notation.management import extract_valid_notation_from_source


from trouver.obsidian.file import MarkdownFile
from trouver.personal_vault.note_processing import process_standard_information_note
from trouver.obsidian.vault import VaultNote

from trouver.machine_learning.notation_summarization import (
    notation_summarization_data_from_note, single_input_for_notation_summarization, NotationSummaryData,
    format_classical, format_training_tokens
)


## TypedDict class for wrapping definition and notation naming dat

In [ ]:
#| export
class DefNotatNamingData(TypedDict):
    def_or_notat: Literal["definition", "notation"] # Whether HTML tag marks an introduced definition or notation.
    name: str # The "name" of the definition or notation.
    text: str # The text in which the definition or notation is introduced. The introduced definition/notation is marked with an HTML tag.
    note_name: str # The name of the VaultNote that `text` comes from.

## Gather ML data from information notes

In [ ]:
#| export

# TODO: test
# TODO: change return type to `DefNotatNamingData`
def def_notat_naming_data_from_information_note(
        info_note: VaultNote, # The standard information note from which to draw data.
    ) -> list[DefNotatNamingData]: # Each dict corresponds to a single datapoint, which holds the data of the naming of a single definition or notation (latex str) introduced in `info_note`. 
    """
    Obtain data for naming definitions and notations for a standard information
    note.

    Definitions and notations should be marked by HTML tags (see
    `machine_learning.tokenize.def_and_notat_token_classification`).
    - A definition is to be marked by an HTML tag with a `definition` attribute,
      which is the definition's "name", i.e. words and/or phrases describing what
      the definition is called and to what objects/situations the definition
      is applicable. If multiple combinations of words/phrases are appropriate,
      then they are separated by a single semicolon `;`. If the `definition`
      attribute is `""`, then the definition name has not been marked, both manually
      and automatically.
    - A notation (technically the full LaTeX string in which the notation is
      introducedis) is to be marked by an HTML tag with a `notation` attribute,
      which is the notation's "name", i.e. the actual notation introduced in
      the LaTeX string (without surrounding dollar signs (`$` or `$$`)). If
      multiple notations are appropriate, then they are separated by
      double semicolons `;;`. If the `notation` attribute is `""`, then it
      means that either the notation has not been marked, or that the
      LaTeX string (minus the surrounding dollar signs) is exactly the
      introduced notation. 


    **Returns**
    - list[dict[str, str]]
        - Each dict corresponds to a single datapoint, which holds the data of
          the naming of a single definition or notation (latex str) introduced
          in `info_note`. The keys are `'text'` and `'definition`' or
          `'notation`'. The `text` entry should be the processed text of
          `info_note`, see `process_standard_information_note` 

    """
    mf = MarkdownFile.from_vault_note(info_note)

    # Processes the info note in all ways except for the HTML tags
    mf = process_standard_information_note(
        mf, info_note.vault,
        True, True, True, True, True, False, True, True, True, True,
        True, True, True, None, True)
    
    text_without_html_tags, removed_tags = remove_html_tags_in_text(str(mf))
    list_of_dicts: list[DefNotatNamingData] = []
    for removed_tag, start, end in removed_tags:
        if 'definition' in removed_tag.attrs:
            def_or_notat = 'definition'
        elif 'notation' in removed_tag.attrs:
            def_or_notat = 'notation'
        else:
            continue
        location_marking_tag = BeautifulSoup(f'<b {def_or_notat}="">{removed_tag.text}', 'html.parser')

        data_point_dict = DefNotatNamingData(
            def_or_notat=def_or_notat,
            name=removed_tag.attrs[def_or_notat],
            text=add_HTML_tag_data_to_raw_text(
                text_without_html_tags, [(location_marking_tag, start, end)]),
            note_name=info_note.name
            )
        list_of_dicts.append(data_point_dict)

    return list_of_dicts

## Augment data

In [ ]:
#| export
def _split_text_by_html_data_parts(
        datapoint: DefNotatNamingData
        ) -> tuple[str, str, str, bs4.element.Tag]: # The text before the HTML tag, the text of the HTML tag, and the text after the HTML tag
    r"""
    Helper function
    """
    to_return: list[str] = []
    html_data: StrAndHTMLTagsWithIndices = remove_html_tags_in_text(datapoint['text'])
    raw_text: str = html_data.raw_text
    tags: list[HTMLTagWithIndices] = html_data.tags
    start, end = tags[0].start, tags[0].end
    
    return (raw_text[:start], raw_text[start:end], raw_text[end:], tags[0].tag)

In [ ]:
#| export
def augment_def_and_notat_naming_data(
        datapoint: DefNotatNamingData,
        num_augmentation_sets: int = 1, # Each augmentation set consists of an augmentation with low, medium, and high probability modifications.
        seed: Optional[int] = None
        ) -> list[DefNotatNamingData]:
    r"""
    Augment a given datapoint for HTML tagging.
    """
    augmented_datapoints: list[DefNotatNamingData] = []
    pieces: tuple[str, str, str, bs4.element.Tag] = _split_text_by_html_data_parts(datapoint)
    if seed is not None:
        random.seed(seed)

    for _ in range(num_augmentation_sets):
        augmented_datapoints.append(
            _augment_def_and_notat_naming_data_once(pieces, 'low', datapoint))
        augmented_datapoints.append(
            _augment_def_and_notat_naming_data_once(pieces, 'mid', datapoint))
        augmented_datapoints.append(
            _augment_def_and_notat_naming_data_once(pieces, 'hi', datapoint))
        # augmented_datapoints.append(_augment_html_data_once(pieces, 'high'))
    return augmented_datapoints


def _augment_def_and_notat_naming_data_once(
        pieces: tuple[str, str, str, bs4.element.Tag],
        modification: Literal['low', 'mid', 'high'],
        original_datapoint: DefNotatNamingData,
        ) -> DefNotatNamingData:

    methods = [
        # (push_dollar_signs,0.2),
        (remove_font_styles_at_random, 0.1), (change_font_styles_at_random, 0.2), (change_greek_letters_at_random, 0.1), 
        (remove_math_keywords,0.1), (random_latex_command_removal,0.2),
        (random_word_removal,0.1), (dollar_sign_manipulation,0.05),
        (random_char_modification,0.001)]
    if modification == 'low':
        method_inclusion_chance = 0.3
        scale = 0.5
    elif modification == 'mid':
        method_inclusion_chance = 0.5
        scale = 1.0
    else:
        method_inclusion_chance = 0.8
        scale = 1.5
    
    random_methods = []
    def create_method(method, p, scale):
        return lambda x: method(x, p=p*scale)
    for method, p in methods:
        if random.random() < method_inclusion_chance:
            random_methods.append(create_method(method, p, scale))

    start_augment = augment_text(pieces[0], random_methods)
    tag = copy.copy(pieces[3])
    mid_augment_with_html_tag = augment_text(pieces[1], random_methods)
    tag.string = mid_augment_with_html_tag
    end_augment = augment_text(pieces[2], random_methods)
    accumulated_text = f'{start_augment}{str(tag)}{end_augment}'
    return DefNotatNamingData(
        def_or_notat=original_datapoint['def_or_notat'],
        name=original_datapoint['name'],
        note_name=original_datapoint['note_name'],
        text=accumulated_text
        )

## Use the ML model

In [ ]:
#| export

# TODO: mark the note with and `_auto` tag and make it so that 
def predict_names(
        info_note: VaultNote,
        def_and_notat_pipeline: Optional[pipelines.text2text_generation.SummarizationPipeline], # A pipeline wrapping an ML model which predicts the naming of both definition and notations.
        def_pipeline: Optional[pipelines.text2text_generation.SummarizationPipeline],  # A pipeline wrapping an ML model which predicts the naming of definitions. 
        notat_pipeline: Optional[pipelines.text2text_generation.SummarizationPipeline], # A pipeline wrapping an ML model which predicts the naming of notations. 
        ) -> list[str]:
    r"""
    Predict the names of the definitions and notations using the trained ML models

    Either `def_and_notat_pipeline` or both `def_pipeline` and `notat_pipeline`
    should be provided.
    """
    if (def_and_notat_pipeline is None and 
            (def_pipeline is None or notat_pipeline is None)):
        raise ValueError(
            "Expected `def_and_notat_pipeline` to be specified or "
            "both `def_pipeline` and `notat_pipeline` to be specified.")
    data_points = def_notat_naming_data_from_information_note(info_note)
    return [_name_prediction_for_data_point(
        data_point, def_and_notat_pipeline, def_pipeline, notat_pipeline)
        for data_point in data_points]


def _name_prediction_for_data_point(
        data_point: DefNotatNamingData, 
        def_and_notat_pipeline: Optional[pipelines.text2text_generation.SummarizationPipeline], 
        def_pipeline: Optional[pipelines.text2text_generation.SummarizationPipeline],  
        notat_pipeline: Optional[pipelines.text2text_generation.SummarizationPipeline], 
        ) -> str:
    if def_and_notat_pipeline is not None:
        summarizer = def_and_notat_pipeline
    elif data_point['def_or_notat'] == 'definition':
    # elif 'definition' in data_point:
        summarizer = def_pipeline
        summarizer_output = summarizer(data_point['text'])
    else:
        summarizer = notat_pipeline
        summarizer_output = summarizer(data_point['text'], max_length=20, min_length=0)
    return summarizer_output[0]['summary_text']


### Combining the steps

In [ ]:
#| export
from typing import List, Optional, Protocol, Any, Union
import warnings

# Assuming these imports exist in your environment based on context
# from your_module import VaultNote, MarkdownFile, pipelines, DefNotatNamingData
# from your_module import remove_html_tags_in_text, add_HTML_tag_data_to_raw_text
# from your_module import predict_names, def_notat_naming_data_from_information_note
# from your_module import math_mode_string_has_soft_or_hard_syntax_errors, extract_valid_notation_from_source, fix_autogen_formatting

class NamingStrategy(Protocol):
    """
    Interface for predicting definition and notation names.
    """
    def predict(self, info_note: 'VaultNote') -> List[str]:
        ...

class Seq2SeqNamingStrategy:
    """
    Strategy using the existing Fine-Tuned T5/Seq2Seq pipelines.
    Wraps the original `predict_names` function.
    """
    def __init__(
        self,
        def_and_notat_pipeline: Optional[Any] = None,
        def_pipeline: Optional[Any] = None,
        notat_pipeline: Optional[Any] = None
    ):
        self.def_and_notat_pipeline = def_and_notat_pipeline
        self.def_pipeline = def_pipeline
        self.notat_pipeline = notat_pipeline

    def predict(self, info_note: 'VaultNote') -> List[str]:
        return predict_names(
            info_note, 
            self.def_and_notat_pipeline, 
            self.def_pipeline, 
            self.notat_pipeline
        )


DEFINITION_NAMING_SYSTEM_PROPT = r"""
### Role
You are a Formal Mathematical Lexicographer. Your goal is to generate a "Fully Parametrized Name" for a definition identified by a `<b>` tag.

### Intent: The Silver Platter
The result must be a "de-referenced" formal title that specifies the **Circumstances of Admissibility**. It must state what the concept is and what ambient environment or prerequisite objects it requires. This ensures a reader knows exactly when the discussion is valid without re-reading the source.

### Operational Logic
1. **The Lead-In Rule**: The output must start with the same word(s) as the text inside the <b> tag. Use the tag text as the "Head" of your phrase. While you may expand variables or adjust grammar within or after this head to satisfy other rules, you must not replace the head with a different concept found elsewhere in the text. Capitalize the first letter of the first word of the output (the Head), even if the text inside the <b> tag is lowercase.
2. **The Adjective-Noun Rule**: If the text in the <b> tag is an adjective (e.g., "ordered"), immediately follow it with the noun it qualifies from the text (e.g., "ordered k-simplex") to complete the "Head."
3. **The Nominal Priority**: If the head is a proper named object (e.g., "n-sphere"), DO NOT append its construction logic or definition (e.g., do not add "defined as the suspension of...").
4. **Identity Replacement**: If the bolded term is a classification for a variable (e.g., "f is an n-equivalence"), do not use "of a [Variable-Type]." Instead, replace the variable's role directly into the head or use a relational preposition.
   - **Correct**: "n-equivalence between spaces" (Relational) or "n-equivalent map" (Integrated).
   - **Incorrect**: "n-equivalence of a map" (Possessive).
5. **Preposition Choice**: Use "between" for maps/morphisms, "on" for operations/structures, "over" for algebraic environments (rings/fields), and "in" for topological or categorical hosting.
6. **The Admissibility Boundary**: 
   - **INCLUDE**: The "Prerequisite Setup" (the objects acted upon, the category/space, and secondary parameters required to initialize the concept).
   - **EXCLUDE**: The "Internal Specification" (specific algebraic formulas, mapping identities, or the local logical predicates that define the term).
   - **GUIDELINE**: State the **Construction** (the "What" and "Where"), not the **Instruction** (the "How").
7. **Formalize Shorthands**: Always expand mathematical shorthands or symbols for categories/spaces (e.g., replace Prof with "the category of profinite sets") to ensure the name is "fully parametrized."
8. **Scope Isolation**: The name must describe the object in the <b> tag and only that object. Do not "borrow" nouns from surrounding sentences (like "covering" or "sheaves") unless they are the direct host structure (the "Where") for the tagged object.
9. **The Ambient Structure Rule**: Identify the "Container" or "Base Structure" that hosts the definition (e.g., a topological space, a category, a base ring, or a manifold). If the text establishes a variable as the host (e.g., "Let X be a space"), include the category of that host in the final name.
   - **Pattern**: [Head] ... [in / over / of / within] a [Host Structure].
   - **Example**: "Freely homotopic loops **in a topological space** along a path."
10. **Environment vs. Building Blocks**: Do not append the components used to build the head. 
   - **Correct**: "$\varepsilon$-hermitian module **over a ring with involution**" (The ring is the ambient stage).
   - **Incorrect**: "(pointed) $n$-sphere **as a suspension of a two-point set**" (The suspension and set are internal building blocks).
11. **Latex Integrity**: Retain all mathematical symbols in their original LaTeX formatting. Revert OCR errors (e.g., "ε") back to proper LaTeX (e.g., "$\varepsilon$").


### Reference Examples: 
The following examples are provided in a redacted format to demonstrate the required mapping between input context and output nomenclature:

- **Input**: Translation 1.2.8 Shifting indices, or <b definition="">translation</b>, is another useful operation we can perform on chain and cochain complexes. If $C$ is a complex and $p$ an integer, we form $C[p]_n=C_{n+p}$...
  **Output**: Translation on chain complexes and cochain complexes
- **Input**: Let $K$ be a geometric simplicial complex... Each $k$-simplex has $k+1$ faces, which are <b definition="">ordered</b> if the set $K_0$ of vertices is ordered...
  **Output**: Ordered $k$-simplex of a geometric simplicial complex
- **Input**: An arcwise connected space $X$ with $\pi_1(X, x_0)=1$ is called <b definition="">"simply connected"</b>.
  **Output**: Simply connected space
- **Input**: Let $p: X \rightarrow Y$ be a covering map. A homeomorphism $D: X \rightarrow X$ which covers the identity... is called a <b definition="">deck transformation</b>...
  **Output**: Deck transformation of a covering map
- **Input**: A prime $\mathfrak{p}$ of an algebraic number field $K$ is a class of equivalent valuations... The nonarchimedean equivalence classes are called <b definition="">finite primes</b>...
  **Output**: Finite prime of an algebraic number field
- **Input**: Let $A$ be a ring with involution and let $\mathfrak{P}-A$ be the category of finitely generated projective right $A$-modules. We call a pair $(M, b)$... an <b definition="">$\varepsilon$-hermitian module</b>.
  **Output**: $\varepsilon$-hermitian module on a finitely generated projective right module over a ring with involution
- **Input**: Let $f: A \rightarrow X$ be a map. Then $f$ is called a <b definition="">cofibration</b> if...
  **Output**: Cofibration between spaces
- **Input**: A map $f: X \rightarrow Y$ is an <b definition="">$n$-equivalence</b> if $f_{\#}$ is an isomorphism for $i < n$...
  **Output**: $n$-equivalence between spaces

### Constraint Checklist
- **NO PREAMBLE/LABELS**: Start the response immediately with the name. 
- **Single Noun Phrase**: Output exactly one formal, grammatically fluid noun phrase.
- **Variable-Free**: Replace all symbols with their conceptual identities from the text.
- **No Definitions or Constructions**: Do not include the internal makeup, construction steps, or words like 'defined as', 'which is', 'as the', 'being', or 'given by'. If the Head is a complete named mathematical object (e.g., $n$-sphere), stop immediately unless it explicitly requires an external ambient environment to be valid.
- **Avoid Possessives for Identity**: If the definition defines what a map or object IS, do not use the word "of" to link the name to its variable's category (e.g., never say "cofibration of a map"; say "cofibration between spaces").

### Input Format
The user will provide a passage of mathematical prose containing a `<b>` tag.
"""


class LMStudioNamingStrategy:
    """
    Strategy using an LM Studio LLM object.
    It utilizes `def_notat_naming_data_from_information_note` to generate 
    discrete data points, ensuring the LLM focuses on specific tags just like the T5 model.
    """
    def __init__(
        self, 
        model: Any, # Type: lmstudio.LLM
        temperature: float = 0.1,
        max_context: int = 4096,
        verbose: bool = False
    ):
        self.model = model
        self.temperature = temperature
        self.max_context = max_context
        self.verbose = verbose
        
        self.def_system_prompt = DEFINITION_NAMING_SYSTEM_PROPT
        self.notat_system_prompt = (
            "You are a mathematician. The user will provide text where a specific "
            "notation is marked with an HTML tag. Provide the NAME of that notation. "
            "Output ONLY the name."
        )

    def predict(self, info_note: 'VaultNote') -> List[str]:
        # 1. Generate Data Points (Reusing existing logic)
        data_points = def_notat_naming_data_from_information_note(info_note)
        predictions = []
        
        # 2. Iterate and Predict
        for i, data_point in enumerate(data_points):
            pred = self._predict_single(data_point)
            predictions.append(pred)
            if self.verbose:
                print(f"[{i+1}/{len(data_points)}] {data_point['def_or_notat']}: {pred}")
                
        return predictions

    def _predict_single(self, data_point: 'DefNotatNamingData') -> str:
        text = data_point['text']
        is_def = data_point['def_or_notat'] == 'definition'
        system_prompt = self.def_system_prompt if is_def else self.notat_system_prompt
        
        # Simple truncation to fit context
        # (Assuming ~3 chars per token as a safe heuristic)
        overhead_chars = len(system_prompt) * 4 + 500
        max_text_chars = (self.max_context * 3) - overhead_chars
        if len(text) > max_text_chars:
            text = text[:max_text_chars]

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Text:\n{text}\n\nOutput Name:"}
        ]

        try:
            # Using lmstudio.LLM.respond
            result = self.model.respond(
                {"messages": messages}, 
                config={"temperature": self.temperature}
            )
            return str(result).strip()
        except Exception as e:
            print(f"LLM Error: {e}")
            return ""


In [ ]:
#| export

from typing import List, Optional, Any, Callable, Dict
import warnings

# Assuming these are your global prompt constants
# from your_module import DEFINITION_NAMING_SYSTEM_PROPT, NOTATION_NAMING_SYSTEM_PROPT 

class UnifiedNamingStrategy:
    def __init__(
        self,
        def_pipeline: Optional[Any] = None,
        def_llm: Optional[Any] = None,
        notat_pipeline: Optional[Any] = None,
        notat_llm: Optional[Any] = None,
        llm_temperature: float = 0.1,
        llm_max_context: int = 4096,
        context: Optional[str] = None,
        verbose: bool = False
    ):
        self.verbose = verbose
        self.llm_temperature = llm_temperature
        self.llm_max_context = llm_max_context
        self.context = context

        # Definition Handler
        if def_llm is not None:
            self._validate_llm(def_llm, "def_llm")
            self.def_handler = self._create_llm_handler(def_llm, "definition")
        elif def_pipeline is not None:
            self.def_handler = self._create_t5_handler(def_pipeline, "definition")
        else:
            self.def_handler = lambda dp: "" 
            
        # Notation Handler
        if notat_llm is not None:
            self._validate_llm(notat_llm, "notat_llm")
            self.notat_handler = self._create_llm_handler(notat_llm, "notation")
        elif notat_pipeline is not None:
            self.notat_handler = self._create_t5_handler(notat_pipeline, "notation")
        else:
            self.notat_handler = lambda dp: ""

    def _validate_llm(self, model: Any, param_name: str):
        if not hasattr(model, 'respond'):
            raise ValueError(f"Object passed to {param_name} must have a .respond() method.")

    def predict(self, info_note: 'VaultNote', overwrite: bool = False) -> List[str]:
        """
        Predicts names. If overwrite is False, skips inference for tags that already have names.
        """
        # 1. Generate Data Points (passing overwrite flag to check existing values)
        data_points = self._get_processed_data_points(info_note, overwrite)
        predictions = []
        
        # 2. Iterate
        for i, data_point in enumerate(data_points):
            # OPTIMIZATION: Check if we should skip prediction
            if not data_point.get('needs_prediction', True):
                # Just return the existing name without calling the model
                pred = data_point['name']
                if self.verbose:
                    print(f"[{i+1}/{len(data_points)}] {data_point['def_or_notat']}: Skipped (Exists: '{pred}')")
            else:
                # Run the actual model
                if data_point['def_or_notat'] == 'definition':
                    if self.verbose:
                        print(f"Predicting on the following input text:\n\n{data_point['text']}\n\n")
                    pred = self.def_handler(data_point)
                    # if self.verbose:
                    #     print(f"Output text:\n\n{pred}\n\n")
                else:
                    pred = self.notat_handler(data_point)
                
                if self.verbose:
                    print(f"[{i+1}/{len(data_points)}] {data_point['def_or_notat']}: Predicted '{pred}'")
            
            predictions.append(pred)
                
        return predictions

    def _get_processed_data_points(self, info_note: 'VaultNote', overwrite: bool) -> List[Dict]:
        mf = MarkdownFile.from_vault_note(info_note)
        # Process note but keep HTML tags to locate them
        mf = process_standard_information_note(mf, info_note.vault, remove_html_tags=False)
        
        processed_text = str(mf)
        text_without_html_tags, tags_and_locats = remove_html_tags_in_text(processed_text)
        
        data_points = []
        for tag, start, end in tags_and_locats:
            if 'definition' in tag.attrs:
                def_or_notat = 'definition'
                existing_name = tag.attrs['definition']
            elif 'notation' in tag.attrs:
                def_or_notat = 'notation'
                existing_name = tag.attrs['notation']
            else:
                continue

            # OPTIMIZATION LOGIC:
            # If we are NOT overwriting, and there is an existing name, 
            # we mark this point as NOT needing prediction.
            needs_prediction = True
            if not overwrite and existing_name.strip() != "":
                needs_prediction = False

            # Create input text for model (only strictly needed if needs_prediction is True, 
            # but we generate it to keep structure consistent)
            location_marking_tag = f'<b {def_or_notat}="">{tag.text}</b>'
            input_text = (
                text_without_html_tags[:start] + 
                location_marking_tag + 
                text_without_html_tags[end:]
            )

            data_points.append({
                'def_or_notat': def_or_notat,
                'text': input_text,
                'name': existing_name,
                'needs_prediction': needs_prediction # <--- New Flag
            })
            
        return data_points

    def _create_llm_handler(self, model: Any, mode: str) -> Callable:
        # FIX: Select prompt based on mode
        if mode == 'definition':
            # Ensure DEFINITION_NAMING_SYSTEM_PROPT is defined/imported
            system_prompt = DEFINITION_NAMING_SYSTEM_PROPT 
        else:
            # Fallback or specific prompt for notations
            system_prompt = (
                "You are a mathematician. The user will provide text where a specific "
                "notation is marked with an HTML tag. Provide the NAME of that notation. "
                "Output ONLY the name."
            )

        def llm_predict(data_point):
            text = data_point['text']
            extra_context = self.context or ""
            

            # Truncation Logic
            # Constants for estimation
            CHARS_PER_TOKEN = 3  # Conservative for LaTeX/Technical text

            # Calculate how many tokens the static parts use (roughly)
            system_prompt_tokens = len(system_prompt) / CHARS_PER_TOKEN
            context_tokens = len(extra_context) / CHARS_PER_TOKEN
            safety_buffer_tokens = 500 # For the LLM's response and overhead

            # Calculate available tokens for the 'text' variable
            available_tokens = self.llm_max_context - system_prompt_tokens - context_tokens - safety_buffer_tokens

            # Convert available tokens back to a character limit for truncation
            available_chars = int(available_tokens * CHARS_PER_TOKEN)

            # Final Safeguard
            if available_chars < 500: 
                available_chars = 500

            if len(text) > available_chars:
                text = text[:available_chars]

            # overhead_chars = len(system_prompt) * 4 + 500
            # context_chars = len(extra_context)
            # available = (self.llm_max_context * 3) - overhead_chars - context_chars
            # if available < 500: available = 500
            # if len(text) > available: text = text[:available]

            if extra_context:
                user_content = f"Context:\n{extra_context}\n\nText:\n{text}\n\nOutput Name:"
            else:
                user_content = f"Text:\n{text}\n\nOutput Name:"

            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content}
            ]
            
            try:
                result = model.respond(
                    {"messages": messages}, 
                    config={"temperature": self.llm_temperature}
                )
                return str(result).strip()
            except Exception as e:
                print(f"LLM Error ({mode}): {e}")
                return ""
                
        return llm_predict

    def _create_t5_handler(self, pipeline: Any, mode: str) -> Callable:
        def t5_predict(data_point):
            text = data_point['text']
            if mode == 'notation':
                output = pipeline(text, max_length=20, min_length=0)
            else:
                output = pipeline(text)
            return output[0]['summary_text']
        return t5_predict


In [ ]:
#| hide
from unittest.mock import patch, MagicMock
import unittest

class TestSeq2SeqNamingStrategy(unittest.TestCase):
    def test_predict_delegates_to_predict_names(self):
        """
        Verifies that the predict method calls the global predict_names function
        with the pipelines stored in the instance.
        """
        # 1. Setup
        mock_note = MagicMock()
        mock_def_pipe = "def_pipe_placeholder"
        mock_notat_pipe = "notat_pipe_placeholder"
        
        strategy = Seq2SeqNamingStrategy(
            def_pipeline=mock_def_pipe,
            notat_pipeline=mock_notat_pipe
        )
        
        # 2. Execution & Verification
        # We patch predict_names where it is used (in the module where Strategy is defined)
        with patch('__main__.predict_names') as mock_predict_names:
            mock_predict_names.return_value = ["Name1", "Name2"]
            
            result = strategy.predict(mock_note)
            
            # Assert return value is passed through
            self.assertEqual(result, ["Name1", "Name2"])
            
            # Assert predict_names was called with correct args
            mock_predict_names.assert_called_once_with(
                mock_note, 
                None, # def_and_notat_pipeline (defaulted to None)
                mock_def_pipe, 
                mock_notat_pipe
            )

# Run the test
unittest.TextTestRunner().run(unittest.TestLoader().loadTestsFromTestCase(TestSeq2SeqNamingStrategy))


.
----------------------------------------------------------------------
Ran 1 test in 0.013s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

In [ ]:
#| hide
class TestLMStudioNamingStrategy(unittest.TestCase):
    def setUp(self):
        # Create a mock LLM object that mimics the lmstudio library interface
        self.mock_llm = MagicMock()
        # Mock tokenize to return a list of dummy tokens (length calculation)
        self.mock_llm.tokenize.return_value = [1] * 10 
        
    def test_predict_iterates_and_calls_llm(self):
        """
        Verifies that the strategy processes each data point individually
        and sends the correct prompt type to the LLM.
        """
        # 1. Setup Data
        mock_note = MagicMock()
        
        # Mock the data extraction function to return two distinct items
        fake_data_points = [
            {'text': 'Context A', 'def_or_notat': 'definition'},
            {'text': 'Context B', 'def_or_notat': 'notation'}
        ]
        
        # Mock the LLM response object
        # The strategy calls str(result).strip(), so we mock __str__
        mock_response_obj = MagicMock()
        mock_response_obj.__str__.return_value = "Predicted Name"
        self.mock_llm.respond.return_value = mock_response_obj

        # 2. Execution
        with patch('__main__.def_notat_naming_data_from_information_note', return_value=fake_data_points):
            strategy = LMStudioNamingStrategy(model=self.mock_llm, verbose=False)
            results = strategy.predict(mock_note)

        # 3. Verification
        
        # Should return a list of predictions
        self.assertEqual(results, ["Predicted Name", "Predicted Name"])
        
        # The LLM should have been called twice
        self.assertEqual(self.mock_llm.respond.call_count, 2)
        
        # Check the first call (Definition)
        call_args_1 = self.mock_llm.respond.call_args_list[0]
        messages_1 = call_args_1[0][0]['messages'] # args[0] is the dict passed to respond
        system_prompt_1 = messages_1[0]['content']
        
        # Verify it used the definition prompt logic
        self.assertIn("definition", system_prompt_1)
        self.assertIn("Context A", messages_1[1]['content'])

        # Check the second call (Notation)
        call_args_2 = self.mock_llm.respond.call_args_list[1]
        messages_2 = call_args_2[0][0]['messages']
        system_prompt_2 = messages_2[0]['content']
        
        # Verify it used the notation prompt logic
        self.assertIn("notation", system_prompt_2)
        self.assertIn("Context B", messages_2[1]['content'])

    def test_truncation_logic(self):
        """
        Verifies that long text is truncated before being sent to the LLM.
        """
        mock_note = MagicMock()
        long_text = "A" * 20000 # Very long string
        fake_data_points = [{'text': long_text, 'def_or_notat': 'definition'}]
        
        # Set a small max_context for testing
        strategy = LMStudioNamingStrategy(model=self.mock_llm, max_context=1000)
        
        with patch('__main__.def_notat_naming_data_from_information_note', return_value=fake_data_points):
            strategy.predict(mock_note)
            
        # Get the text actually sent to the LLM
        call_args = self.mock_llm.respond.call_args
        sent_messages = call_args[0][0]['messages']
        sent_user_text = sent_messages[1]['content']
        
        # Assert the text in the message is shorter than the original input
        self.assertTrue(len(sent_user_text) < len(long_text))
        self.assertIn("Text:", sent_user_text)

# Run the tests
unittest.TextTestRunner().run(unittest.TestLoader().loadTestsFromTestCase(TestLMStudioNamingStrategy))


F.
FAIL: test_predict_iterates_and_calls_llm (__main__.TestLMStudioNamingStrategy.test_predict_iterates_and_calls_llm)
Verifies that the strategy processes each data point individually
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\hyunj\AppData\Local\Temp\ipykernel_14120\3057927115.py", line 49, in test_predict_iterates_and_calls_llm
    self.assertIn("Context A", messages_1[1]['content'])
AssertionError: 'Context A' not found in 'Text:\n\n\nOutput Name:'

----------------------------------------------------------------------
Ran 2 tests in 0.024s

FAILED (failures=1)


<unittest.runner.TextTestResult run=2 errors=0 failures=1>

In [ ]:
#| hide
class TestNamingStrategyFactory(unittest.TestCase):
    def setUp(self):
        self.mock_note = MagicMock()
        # Mock the note text to avoid parsing errors
        self.mock_note.text.return_value = "Some text without tags" 
        
    def test_initializes_lmstudio_strategy(self):
        """
        Verifies that passing `llm_model` initializes LMStudioNamingStrategy.
        """
        mock_llm = MagicMock()
        
        # We patch the class itself to check if it gets instantiated
        with patch('__main__.LMStudioNamingStrategy') as MockLMStrategy, \
             patch('__main__.remove_html_tags_in_text', return_value=("text", [])), \
             patch('__main__.MarkdownFile'):
            
            # Mock the predict method so the function completes
            MockLMStrategy.return_value.predict.return_value = []
            
            add_names_to_html_tags_in_info_note(
                self.mock_note,
                llm_model=mock_llm,
                verbose=True # Should be passed to kwargs
            )
            
            # Assert the class was initialized
            MockLMStrategy.assert_called_once()
            
            # Assert arguments were passed correctly
            _, kwargs = MockLMStrategy.call_args
            self.assertEqual(kwargs['model'], mock_llm)
            self.assertEqual(kwargs['verbose'], True)

    def test_initializes_seq2seq_strategy(self):
        """
        Verifies that passing pipelines initializes Seq2SeqNamingStrategy.
        """
        mock_pipe = MagicMock()
        
        with patch('__main__.Seq2SeqNamingStrategy') as MockSeq2SeqStrategy, \
             patch('__main__.remove_html_tags_in_text', return_value=("text", [])), \
             patch('__main__.MarkdownFile'):
            
            MockSeq2SeqStrategy.return_value.predict.return_value = []
            
            add_names_to_html_tags_in_info_note(
                self.mock_note,
                def_pipeline=mock_pipe,
                notat_pipeline=mock_pipe
            )
            
            # Assert the class was initialized
            MockSeq2SeqStrategy.assert_called_once()
            
            # Assert arguments were passed correctly
            _, kwargs = MockSeq2SeqStrategy.call_args
            self.assertEqual(kwargs['def_pipeline'], mock_pipe)
            self.assertEqual(kwargs['notat_pipeline'], mock_pipe)

    def test_raises_error_if_no_strategy_provided(self):
        """
        Verifies that a ValueError is raised if no valid arguments are provided.
        """
        with self.assertRaises(ValueError):
            add_names_to_html_tags_in_info_note(self.mock_note)

# Run the tests
unittest.TextTestRunner().run(unittest.TestLoader().loadTestsFromTestCase(TestNamingStrategyFactory))


EF.
ERROR: test_initializes_lmstudio_strategy (__main__.TestNamingStrategyFactory.test_initializes_lmstudio_strategy)
Verifies that passing `llm_model` initializes LMStudioNamingStrategy.
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\hyunj\AppData\Local\Temp\ipykernel_14120\2762704183.py", line 22, in test_initializes_lmstudio_strategy
    add_names_to_html_tags_in_info_note(
  File "C:\Users\hyunj\AppData\Local\Temp\ipykernel_14120\100205915.py", line 141, in add_names_to_html_tags_in_info_note
    raise ValueError(
ValueError: You must provide a handler for both definitions and notations.

FAIL: test_initializes_seq2seq_strategy (__main__.TestNamingStrategyFactory.test_initializes_seq2seq_strategy)
Verifies that passing pipelines initializes Seq2SeqNamingStrategy.
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\hyunj\AppData\Local\Tem

<unittest.runner.TextTestResult run=3 errors=1 failures=1>

In [ ]:
#| export

def add_names_to_html_tags_in_info_note(
        info_note: 'VaultNote',
        # Strategy Args
        def_pipeline: Optional[Any] = None,
        def_llm_model: Optional[Any] = None,
        notat_pipeline: Optional[Any] = None,
        notat_llm_model: Optional[Any] = None,
        naming_strategy: Optional[UnifiedNamingStrategy] = None,
        llm_max_context: int = 4096,
        context: Optional[str] = None,
        # Config Args
        overwrite: bool = False, 
        fix_formatting: bool = True, 
        correct_syntax: bool = True, 
        verbose: bool = False,
        **strategy_kwargs
        ) -> None:
    
    # 1. Resolve Strategy
    strategy: UnifiedNamingStrategy
    if naming_strategy is not None:
        strategy = naming_strategy
        if context is not None: strategy.context = context
    else:
        # Check for valid configuration
        has_def = def_pipeline or def_llm_model
        has_notat = notat_pipeline or notat_llm_model
        if not (has_def and has_notat):
             raise ValueError("Provide handlers for both definitions and notations.")

        strategy = UnifiedNamingStrategy(
            def_pipeline=def_pipeline,
            def_llm=def_llm_model,
            notat_pipeline=notat_pipeline,
            notat_llm=notat_llm_model,
            llm_max_context=llm_max_context,
            context=context,
            verbose=verbose,
            **strategy_kwargs
        )

    # 2. Execution
    # Note: We use raw text here to ensure we can write back to the file correctly.
    # The strategy internally processes the text for the model's benefit.
    raw_info_note_text = info_note.text()
    raw_info_note_text_minus_html_tags, tags_and_locats = remove_html_tags_in_text(
        raw_info_note_text)
    
    predicted_names = strategy.predict(info_note, overwrite=overwrite)

    # 3. Validation & Writing
    if len(predicted_names) != len(tags_and_locats):
        warnings.warn(
            f"Mismatch in {info_note.name}: Found {len(tags_and_locats)} tags in raw text, "
            f"but generated {len(predicted_names)} predictions from processed text.\n"
            "This usually happens if tags are inside comments (which are removed during processing)."
        )
        min_len = min(len(predicted_names), len(tags_and_locats))
        predicted_names = predicted_names[:min_len]
        tags_and_locats = tags_and_locats[:min_len]

    new_tags_and_locations = []
    any_preds_written = False
    
    for name, (tag, start, end) in zip(predicted_names, tags_and_locats):
        if 'definition' in tag.attrs:
            def_or_notat = 'definition'
        elif 'notation' in tag.attrs:
            def_or_notat = 'notation'
            if correct_syntax and math_mode_string_has_soft_or_hard_syntax_errors(name):
                name = extract_valid_notation_from_source(name, tag.text)
            if fix_formatting:
                name = fix_autogen_formatting(name)
        else:
            def_or_notat = ''
            
        if def_or_notat and (tag.attrs[def_or_notat] == "" or overwrite):
            tag[def_or_notat] = name
            any_preds_written = True
        new_tags_and_locations.append((tag, start, end))
        
    new_info_note_text = add_HTML_tag_data_to_raw_text(
        raw_info_note_text_minus_html_tags, new_tags_and_locations)
        
    mf = MarkdownFile.from_string(new_info_note_text)
    if any_preds_written:
        mf.add_tags('_auto/def_and_notat_names_added')
    mf.write(info_note)


In [ ]:
# # Setup the strategy
# t5_strategy = Seq2SeqNamingStrategy(
#     def_and_notat_pipeline=my_pipeline
# )

# # Run function
# add_names_to_html_tags_in_info_note(
#     info_note=my_note,
#     naming_strategy=t5_strategy
# )

# # Setup the strategy
# lm_strategy = LMStudioNamingStrategy(
#     base_url="http://localhost:1234/v1",
#     system_prompt="You are a math expert. Extract definition names..."
# )

# # Run function
# add_names_to_html_tags_in_info_note(
#     info_note=my_note,
#     naming_strategy=lm_strategy
# )


In [ ]:
#| export
# def add_names_to_html_tags_in_info_note(
#         info_note: VaultNote,
#         def_and_notat_pipeline: Optional[pipelines.text2text_generation.SummarizationPipeline] = None, # A pipeline wrapping an ML model which predicts the naming of both definition and notations.
#         def_pipeline: Optional[pipelines.text2text_generation.SummarizationPipeline] = None,  # A pipeline wrapping an ML model which predicts the naming of definitions. 
#         notat_pipeline: Optional[pipelines.text2text_generation.SummarizationPipeline] = None, # A pipeline wrapping an ML model which predicts the naming of notations. 
#         # summarizer: pipelines.text2text_generation.SummarizationPipeline, # The pipeline with the ML model
#         overwrite: bool = False, # If `True`, overwrite pre-existing, nonempty attributes. If `False`, ignore pre-existing, nonempty attributes and only write on attributes that are empty.
#         fix_formatting: bool = True, # If `True`, fix the formatting for notation names.
#         correct_syntax: bool = True, # If `True`, attempt to fix syntax errors for notation names.
#         ) -> None:
#     """
#     Predict the names of definitions and notations marked with
#     HTML tags within `info_note` and write those names in the
#     `"definition"` or `"notation"` attributes in each tag.

#     Either `def_and_notat_pipeline` or both `def_pipeline` and `notat_pipeline`
#     should be provided.

#     An `#_auto/notation_notes_linked` tag is added to
#     `origin_notation_note` if such a tag is not already
#     present.
#     """
#     raw_info_note_text = info_note.text()
#     raw_info_note_text_minus_html_tags, tags_and_locats = remove_html_tags_in_text(
#         raw_info_note_text)
#     predicted_names = predict_names(
#         info_note, def_and_notat_pipeline, def_pipeline,
#         notat_pipeline)

#     # If somehow a different number of HTML tags were found
#     if len(predicted_names) != len(tags_and_locats):
#         # TODO: do warning
#         warnings.warn(
#             "Somehow, an inconsistent number of HTML tags are "
#             f"detected in the note: {info_note.name}.\n"
#             "This will raise some indexing issues when marking the definition "
#             "and notation names")
#     new_tags_and_locations = []
#     any_preds_written = False
#     for name, (tag, start, end) in zip(predicted_names, tags_and_locats):
#         if 'definition' in tag.attrs:
#             def_or_notat = 'definition'
#         elif 'notation' in tag.attrs:
#             def_or_notat = 'notation'
#             if correct_syntax and math_mode_string_has_soft_or_hard_syntax_errors(name):
#                 name = extract_valid_notation_from_source(name, tag.text)
#             if fix_formatting:
#                 name = fix_autogen_formatting(name)
#         else:
#             # tag could be neither a definition nor a notation tag.
#             def_or_notat = ''
#         if def_or_notat and (tag.attrs[def_or_notat] == "" or overwrite):
#             tag[def_or_notat] = name
#             any_preds_written = True
#         new_tags_and_locations.append((tag, start, end))
#     new_info_note_text = add_HTML_tag_data_to_raw_text(
#         raw_info_note_text_minus_html_tags, new_tags_and_locations)
#     mf = MarkdownFile.from_string(new_info_note_text)
#     if any_preds_written:
#         mf.add_tags('_auto/def_and_notat_names_added')
#     mf.write(info_note)


# # def _correct_syntax(
# #         name: str,
# #         tag: Tag
# #         ) -> str:
# #     """
# #     This is a helper function of `add_names_to_html_tags_in_info_note`.
# #     """
# #     replacement_candidates = _list_of_candidates_from_math_mode_strings(tag.text)
# #     return correct_latex_syntax_error(name, replacement_candidates)

In [ ]:
from unittest.mock import patch, MagicMock

# 1. Update MockVaultNote to satisfy the interface expected by MarkdownFile.write
class MockVaultNote:
    """
    Mocks the VaultNote object with all necessary methods.
    """
    def __init__(self, html_content: str):
        self._content = html_content
        self.name = "Mock Note" 
        self.vault = MagicMock()

    def text(self) -> str:
        """Returns the current HTML content."""
        return self._content

    def replace_text(self, new_text: str):
        """Updates the HTML content."""
        self._content = new_text
        
    def path(self):
        return "mock_note.md" 
    
    # --- Methods required by MarkdownFile.write ---
    def exists(self) -> bool:
        return True 
        
    def create(self):
        pass 

def test_extract_valid_notation_from_source_is_called():
    """
    Verifies that `extract_valid_notation_from_source` is called with the correct arguments.
    """
    # Setup Data
    html = r'The homology is <span notation="" style="border-width:1px">$$ H_{*}(X ; M)=H_{*}(S(X) \otimes M) $$</span>.'
    note = MockVaultNote(html)
    bad_prediction = r"H_{* ; M)" 

    # Setup Strategy Mock
    mock_strategy = MagicMock()
    mock_strategy.predict.return_value = [bad_prediction]

    # Patch dependencies
    # NOTE: We patch '__main__.MarkdownFile' because in a notebook, the function 
    # usually holds a reference to the class imported into __main__.
    with patch('__main__.extract_valid_notation_from_source', return_value="dummy_corrected_value") as mock_extract, \
         patch('__main__.math_mode_string_has_soft_or_hard_syntax_errors', return_value=True), \
         patch('__main__.MarkdownFile') as MockMarkdownFile:

        # Setup the MockMarkdownFile instance so .write() doesn't crash
        mock_mf_instance = MockMarkdownFile.from_string.return_value
        
        # Run function
        add_names_to_html_tags_in_info_note(
            note, 
            naming_strategy=mock_strategy,
            correct_syntax=True
        )

        # Verification
        mock_extract.assert_called_once()

        args, _ = mock_extract.call_args
        actual_prediction_arg = args[0]
        actual_source_text_arg = args[1]

        assert actual_prediction_arg == bad_prediction
        assert r"$$ H_{*}(X ; M)=H_{*}(S(X) \otimes M) $$" in actual_source_text_arg

        print("Test Passed: extract_valid_notation_from_source was called correctly.")

# Run the test
test_extract_valid_notation_from_source_is_called()


Test Passed: extract_valid_notation_from_source was called correctly.


In [ ]:
#| hide


# from unittest.mock import patch, MagicMock

# class MockVaultNote:
#     """
#     Mocks the VaultNote object with all necessary methods for 
#     add_names_to_html_tags_in_info_note and MarkdownFile.write.
#     """
#     def __init__(self, html_content: str):
#         self._content = html_content
#         self.name = "Mock Note" 

#     def text(self) -> str:
#         """Returns the current HTML content."""
#         return self._content

#     def replace_text(self, new_text: str):
#         """Updates the HTML content."""
#         self._content = new_text
        
#     # --- Methods required by MarkdownFile.write ---
#     def exists(self) -> bool:
#         return True # Pretend the file exists
        
#     def create(self):
#         pass # Do nothing
        
#     def path(self):
#         return "mock_note.md" # Dummy path


# def test_extract_valid_notation_from_source_is_called():
#     """
#     Verifies that `extract_valid_notation_from_source` is called with the correct arguments.
#     This is a "spy" test to check the wiring.
#     """
#     html = r'The homology is <span notation="" style="border-width:1px;border-style:solid;padding:3px">$$ H_{*}(X ; M)=H_{*}(S(X) \otimes M) $$</span>.'
#     note = MockVaultNote(html)
#     bad_predictions = [r"H_{* ; M)"]

#     # We patch the function we want to spy on.
#     # Replace 'your_module_name.extract_valid_notation_from_source' with the actual import path.
#     with patch('__main__.extract_valid_notation_from_source', return_value="dummy_value") as mock_extract_valid_notation_from_source, \
#          patch('__main__.predict_names', return_value=bad_predictions), \
#          patch('trouver.obsidian.file.MarkdownFile.write'):

#         add_names_to_html_tags_in_info_note(note, def_and_notat_pipeline=None)

#         # --- Verification ---
#         # 1. Assert that the function was called at least once.
#         mock_extract_valid_notation_from_source.assert_called()

#         # 2. Assert it was called with the expected arguments.
#         #    - The bad prediction from the mocked model.
#         #    - The inner text of the HTML tag.
#         expected_prediction = r"H_{* ; M)"
#         expected_source_text = r"$$ H_{*}(X ; M)=H_{*}(S(X) \otimes M) $$"
#         expected_tag = r'<span notation="" style="border-width:1px;border-style:solid;padding:3px">$$ H_{*}(X ; M)=H_{*}(S(X) \otimes M) $$</span>'
        
#         # 1. Get the arguments from the last call
#         # args[0] is expected_prediction, args[1] is the tag object
#         args, kwargs = mock_extract_valid_notation_from_source.call_args
#         actual_prediction = args[0]
#         actual_source = args[1]

#         # 2. Assertions
#         assert actual_prediction == expected_prediction

#         # If 'actual_tag' is a BeautifulSoup Tag, use .get_text() or .string
#         # If it's just a string you want to verify contains something:
#         assert expected_source_text in actual_source

# # Run this test in your notebook. If it passes, the problem is inside extract_valid_notation_from_source.
# test_extract_valid_notation_from_source_is_called()


# Naming notation notes

Another convenient functionality is to name notation notes automatically.

In [ ]:
#| export

# TODO: test
def autogen_name_from_notation_note(
        notation_note: VaultNote, pipeline):
    data_dict: NotationSummaryData = notation_summarization_data_from_note(
        notation_note, notation_note.vault,
        check_for_actual_summarization=False)
    if data_dict is None:
        return None
    # TODO: change formatter to format_training_tokens after retraining the model for this formatting
    input = single_input_for_notation_summarization(
        data_dict, 
        input_formatter=format_classical
    )

    # input = single_input_for_notation_summarization(
    #     data_dict, classical_formatting=True)
    return pipeline(input)[0]['summary_text']

def sanitize_autogen_name(autogen_name):
    autogen_name = autogen_name.replace(' ', '')
    autogen_name = autogen_name.replace('[', '')
    autogen_name = autogen_name.replace(']', '')
    autogen_name = autogen_name.replace('.', '')
    return sanitize_filename(autogen_name)


def add_autogen_name_to_notation_note(
        notation_note: VaultNote,
        autogen_name: str
        ) -> None:
    mf = MarkdownFile.from_vault_note(notation_note)
    if not mf.has_metadata():
        mf.add_metadata_section()

    metadata = mf.metadata()
    metadata['autogen_name'] = [autogen_name] 
    mf.replace_metadata(metadata, enquote_entries_in_fields=['latex_in_original'])

    mf.write(notation_note)
    # mf.metadata



In [ ]:
#| export

# TODO: test
def predict_name_and_add_to_notation_note(
        notation_note: VaultNote,
        notation_note_naming_pipeline: pipelines.text2text_generation.SummarizationPipeline,
        reference: str
        ) -> None:
    """
    Predict an appropriate name for the notation note and add it in the YAML frontmatter metadata.
    """
    mf = MarkdownFile.from_vault_note(notation_note)
    if mf.has_tag('_meta/notation_note_named') or 'autogen_name' in str(mf):
        return
    autogen_name = autogen_name_from_notation_note(notation_note, notation_note_naming_pipeline)
    if autogen_name is None:
        return
    autogen_name = sanitize_autogen_name(autogen_name)
    autogen_name = f'{reference}_notation_{autogen_name}'
    add_autogen_name_to_notation_note(notation_note, autogen_name)